# Turkish Legal Reranker Baseline

## Purpose

This notebook implements a cross-encoder reranking stage that refines hybrid retrieval results.

**Architecture:**
```
Question → Dense+BM25 Hybrid Search → Cross-Encoder Reranking → Top-K Results
```

**What this notebook does:**
- Loads the retrieval corpus and hybrid retrieval artifacts
- Rebuilds hybrid retrieval to get candidate chunks
- Loads a cross-encoder model (MS MARCO-tuned)
- Rescores hybrid candidates using cross-encoder
- Compares results across dense, BM25, hybrid, and reranked stages
- Tests reranking on Turkish legal queries
- Saves reranked results

**What this notebook does NOT do:**
- Generate answers using an LLM
- Evaluate retrieval performance formally
- Fine-tune the reranker model

**Note on Model:**
The cross-encoder (MS MARCO MiniLM) is trained on English MS MARCO dataset for general relevance ranking.
It is NOT Turkish-legal-specific. Future work can replace it with a strong multilingual or domain-adapted reranker.

**Reranking Flow:**
1. Get candidate chunks via hybrid retrieval (size configurable)
2. Build (query, chunk_text) pairs
3. Score with cross-encoder → get relevance scores
4. Sort by cross-encoder score descending
5. Return top-K reranked results

## 1. Environment Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

## 2. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted")

## 3. Configuration and Paths

In [ ]:
# Project configuration
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"
RETRIEVAL_DATA_PATH = f"{PROJECT_ROOT}/data/retrieval/retrieval_corpus_full.csv"
DENSE_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/dense_retrieval"
HYBRID_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/hybrid_retrieval"
RERANKER_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/reranker"

# Model configuration
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Retrieval and reranking configuration
TOP_K_FINAL = 5                  # Final number of reranked results to return
DENSE_CANDIDATES = 20            # Candidates from dense search
BM25_CANDIDATES = 20             # Candidates from BM25 search
HYBRID_CANDIDATES = 20           # Size of hybrid candidate pool before reranking
ALPHA = 0.5                      # Weight for hybrid: alpha*dense + (1-alpha)*bm25

# Create output directory
Path(RERANKER_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Retrieval corpus: {RETRIEVAL_DATA_PATH}")
print(f"Output directory: {RERANKER_OUTPUT_DIR}")
print(f"\nConfiguration:")
print(f"  - Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"  - Reranker model: {RERANKER_MODEL_NAME}")
print(f"  - Top K final: {TOP_K_FINAL}")
print(f"  - Hybrid candidate pool: {HYBRID_CANDIDATES}")
print(f"  - Dense candidates: {DENSE_CANDIDATES}")
print(f"  - BM25 candidates: {BM25_CANDIDATES}")

## 4. Install and Import Dependencies

In [ ]:
import subprocess
import sys

print("Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank-bm25"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
print("✓ Dependencies installed")

In [ ]:
!pip install -U sentence-transformers

In [ ]:
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

print("✓ All imports successful")

## 5. Load Retrieval Corpus and Dense Artifacts

In [ ]:
# Load retrieval corpus
print(f"Loading retrieval corpus...")
df_corpus = pd.read_csv(RETRIEVAL_DATA_PATH)
print(f"✓ Loaded retrieval corpus")
print(f"  - Shape: {df_corpus.shape}")
print(f"  - Non-null chunk_text: {df_corpus['chunk_text'].notna().sum()}")

In [ ]:
# Load FAISS index
index_path = f"{DENSE_OUTPUT_DIR}/faiss_index.bin"
print(f"Loading FAISS index...")
faiss_index = faiss.read_index(index_path)
print(f"✓ FAISS index loaded")
print(f"  - Number of vectors: {faiss_index.ntotal}")
print(f"  - Vector dimension: {faiss_index.d}")

In [ ]:
# Load dense metadata
metadata_path = f"{DENSE_OUTPUT_DIR}/dense_retrieval_metadata.json"
with open(metadata_path, 'r', encoding='utf-8') as f:
    dense_metadata = json.load(f)

print(f"Dense retrieval metadata:")
print(f"  - Model: {dense_metadata['embedding_model']}")
print(f"  - Dimension: {dense_metadata['embedding_dimension']}")

In [ ]:
# Load row mapping for FAISS indexing
mapping_path = f"{DENSE_OUTPUT_DIR}/retrieval_row_mapping.csv"
df_mapping = pd.read_csv(mapping_path)
print(f"✓ Row mapping loaded")
print(f"  - Mapping entries: {len(df_mapping)}")

## 6. Rebuild Hybrid Retrieval Pipeline (Reuse Logic)

In [ ]:
# Load embedding model
print(f"Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL_NAME}")

In [ ]:
def preprocess_text(text: str) -> List[str]:
    """
    Preprocess text for BM25:
    - lowercase
    - strip whitespace
    - collapse repeated spaces
    - tokenize on whitespace
    """
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = " ".join(text.split())
    tokens = text.split()
    return tokens


print("Building BM25 index...")
corpus_tokens = [preprocess_text(text) for text in df_corpus['chunk_text']]
bm25_index = BM25Okapi(corpus_tokens)

print(f"✓ BM25 index built")
print(f"  - Corpus size: {len(corpus_tokens)}")
print(f"  - Vocabulary size: {len(bm25_index.idf)}")

In [ ]:
def retrieve_dense_candidates(
    query: str,
    model,
    index: faiss.Index,
    corpus_df: pd.DataFrame,
    top_k: int = DENSE_CANDIDATES
) -> Dict[str, Tuple[float, int]]:
    """Retrieve candidate chunks using dense retrieval."""
    query_embedding = model.encode(query, normalize_embeddings=True)
    query_embedding = np.array([query_embedding], dtype=np.float32)
    scores, faiss_indices = index.search(query_embedding, top_k)
    
    results = {}
    for faiss_idx, score in zip(faiss_indices[0], scores[0]):
        corpus_idx = df_mapping.iloc[faiss_idx]['corpus_row_index']
        chunk_id = df_mapping.iloc[faiss_idx]['chunk_id']
        results[chunk_id] = (float(score), int(corpus_idx))
    
    return results


def retrieve_bm25_candidates(
    query: str,
    bm25: BM25Okapi,
    corpus_df: pd.DataFrame,
    top_k: int = BM25_CANDIDATES
) -> Dict[str, Tuple[float, int]]:
    """Retrieve candidate chunks using BM25 retrieval."""
    query_tokens = preprocess_text(query)
    bm25_scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_k]
    
    results = {}
    for corpus_idx in top_indices:
        score = float(bm25_scores[corpus_idx])
        if corpus_idx < len(corpus_df):
            chunk_id = corpus_df.iloc[corpus_idx]['chunk_id']
            results[chunk_id] = (score, corpus_idx)
    
    return results


print("✓ Dense and BM25 retrieval functions defined")

In [ ]:
def retrieve_hybrid_candidates(
    query: str,
    model,
    faiss_index: faiss.Index,
    bm25: BM25Okapi,
    corpus_df: pd.DataFrame,
    top_k: int = HYBRID_CANDIDATES,
    alpha: float = ALPHA,
    dense_candidates: int = DENSE_CANDIDATES,
    bm25_candidates: int = BM25_CANDIDATES
) -> pd.DataFrame:
    """
    Retrieve hybrid candidates (not final top-k yet).
    Returns DataFrame with hybrid_score, dense_score, bm25_score for each candidate.
    """
    # Get candidates from both methods
    dense_results = retrieve_dense_candidates(
        query, model, faiss_index, corpus_df, dense_candidates
    )
    bm25_results = retrieve_bm25_candidates(
        query, bm25, corpus_df, bm25_candidates
    )
    
    # Collect all unique chunks
    all_chunks = set(dense_results.keys()) | set(bm25_results.keys())
    
    # Normalize scores
    dense_scores = [score for score, _ in dense_results.values()]
    bm25_scores = [score for score, _ in bm25_results.values()]
    
    dense_min, dense_max = (min(dense_scores) if dense_scores else 0), (max(dense_scores) if dense_scores else 1)
    bm25_min, bm25_max = (min(bm25_scores) if bm25_scores else 0), (max(bm25_scores) if bm25_scores else 1)
    
    dense_range = dense_max - dense_min if dense_max > dense_min else 1
    bm25_range = bm25_max - bm25_min if bm25_max > bm25_min else 1
    
    # Compute hybrid scores for all candidates
    hybrid_scores = {}
    for chunk_id in all_chunks:
        if chunk_id in dense_results:
            dense_score, corpus_idx = dense_results[chunk_id]
            dense_norm = (dense_score - dense_min) / dense_range if dense_range > 0 else 0
        else:
            dense_norm = 0
            corpus_idx = bm25_results[chunk_id][1]
        
        if chunk_id in bm25_results:
            bm25_score, _ = bm25_results[chunk_id]
            bm25_norm = (bm25_score - bm25_min) / bm25_range if bm25_range > 0 else 0
        else:
            bm25_norm = 0
        
        hybrid_score = alpha * dense_norm + (1 - alpha) * bm25_norm
        
        hybrid_scores[chunk_id] = {
            'hybrid_score': hybrid_score,
            'dense_score': dense_results.get(chunk_id, (0, corpus_idx))[0],
            'bm25_score': bm25_results.get(chunk_id, (0, corpus_idx))[0],
            'corpus_idx': corpus_idx
        }
    
    # Sort by hybrid score and take top-k candidates
    sorted_chunks = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1]['hybrid_score'],
        reverse=True
    )[:top_k]
    
    # Build result dataframe
    results = []
    for rank, (chunk_id, scores) in enumerate(sorted_chunks, 1):
        corpus_idx = scores['corpus_idx']
        row = corpus_df.iloc[corpus_idx].to_dict()
        row['rank'] = rank
        row['hybrid_score'] = scores['hybrid_score']
        row['dense_score'] = scores['dense_score']
        row['bm25_score'] = scores['bm25_score']
        results.append(row)
    
    return pd.DataFrame(results)


print("✓ Hybrid retrieval function defined")

## 7. Load Reranker Model

In [ ]:
print(f"Loading cross-encoder reranker model...")
reranker = CrossEncoder(RERANKER_MODEL_NAME)
print(f"✓ Reranker model loaded: {RERANKER_MODEL_NAME}")
print(f"  Note: This model is tuned on English MS MARCO dataset.")
print(f"  It is NOT Turkish-legal-specific. Can be replaced with stronger domain-adapted model.")

## 8. Implement Reranking Function

In [ ]:
def rerank_candidates(
    query: str,
    candidate_df: pd.DataFrame,
    reranker: CrossEncoder,
    top_k: int = TOP_K_FINAL
) -> pd.DataFrame:
    """
    Rerank candidates using cross-encoder.
    
    Args:
        query: search query
        candidate_df: DataFrame with candidates, must have 'chunk_text' column
        reranker: CrossEncoder model
        top_k: number of results to return
    
    Returns:
        DataFrame with reranked results, sorted by reranker_score descending
    """
    if len(candidate_df) == 0:
        return pd.DataFrame()
    
    # Build (query, chunk_text) pairs
    chunks = candidate_df['chunk_text'].tolist()
    pairs = [[query, chunk] for chunk in chunks]
    
    # Score with cross-encoder
    scores = reranker.predict(pairs)
    
    # Add scores to dataframe
    result_df = candidate_df.copy()
    result_df['reranker_score'] = scores
    
    # Sort by reranker score descending
    result_df = result_df.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    
    # Update rank
    result_df['rank'] = range(1, len(result_df) + 1)
    
    # Return top-k
    return result_df.head(top_k)


print("✓ Reranking function defined")

## 9. Test Queries

In [ ]:
# Define test queries
test_queries = [
    "Haksız zenginleşme ile ilgili hükümler nelerdir?",
    "Miras bırakanın tasarruf özgürlüğü nasıl sınırlandırılır?",
    "Ceza muhakemesinde tutuklama şartları nelerdir?",
    "Anayasa'ya göre devletin şekli nedir?",
    "Aile yurdu ile ilgili malik üzerindeki sınırlamalar nelerdir?"
]

print(f"Defined {len(test_queries)} test queries")

## 10. Run Reranking on Test Queries

In [ ]:
all_reranked_results = []
comparison_results = []

print("="*80)
print("RUNNING RERANKER TEST QUERIES")
print("="*80)

for query_idx, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"Query {query_idx}: {query}")
    print(f"{'='*80}")
    
    # Get hybrid candidates
    hybrid_candidates = retrieve_hybrid_candidates(
        query=query,
        model=embedding_model,
        faiss_index=faiss_index,
        bm25=bm25_index,
        corpus_df=df_corpus,
        top_k=HYBRID_CANDIDATES,
        alpha=ALPHA,
        dense_candidates=DENSE_CANDIDATES,
        bm25_candidates=BM25_CANDIDATES
    )
    
    print(f"\nHybrid candidates generated: {len(hybrid_candidates)}")
    
    # Rerank
    reranked = rerank_candidates(
        query=query,
        candidate_df=hybrid_candidates,
        reranker=reranker,
        top_k=TOP_K_FINAL
    )
    
    # Add query info
    reranked['test_query_index'] = query_idx
    reranked['test_query_text'] = query
    all_reranked_results.append(reranked)
    
    # Print detailed comparison
    print(f"\nTop {TOP_K_FINAL} Reranked Results:")
    for _, row in reranked.iterrows():
        print(f"\n  [{row['rank']}] Reranker: {row['reranker_score']:.4f}")
        print(f"      Hybrid: {row['hybrid_score']:.4f} | Dense: {row['dense_score']:.4f} | BM25: {row['bm25_score']:.4f}")
        print(f"      Source: {row['source']}")
        print(f"      Chunk: {row['chunk_text'][:120]}...")
    
    # Show hybrid top-3 for comparison
    print(f"\nPreviously (Hybrid Top 3):")
    for idx, (_, row) in enumerate(hybrid_candidates.head(3).iterrows(), 1):
        print(f"  {idx}. Hybrid: {row['hybrid_score']:.4f} | {row['chunk_text'][:80]}...")
    
    # Store comparison data
    if len(hybrid_candidates) > 0:
        hybrid_top1 = hybrid_candidates.iloc[0]
        reranked_top1 = reranked.iloc[0] if len(reranked) > 0 else None
        
        if reranked_top1 is not None:
            comparison_results.append({
                'query_idx': query_idx,
                'query_text': query,
                'hybrid_top1_source': hybrid_top1['source'],
                'reranked_top1_source': reranked_top1['source'],
                'hybrid_top1_chunk': hybrid_top1['chunk_text'][:150],
                'reranked_top1_chunk': reranked_top1['chunk_text'][:150],
                'hybrid_top1_score': hybrid_top1['hybrid_score'],
                'reranked_top1_score': reranked_top1['reranker_score']
            })

In [ ]:
# Combine all reranked results
df_reranked = pd.concat(all_reranked_results, ignore_index=True)

print(f"\n✓ Completed all reranking queries")
print(f"  - Total reranked results: {len(df_reranked)}")
print(f"  - Results per query: {len(df_reranked) // len(test_queries)}")

## 11. Comparison: Dense vs BM25 vs Hybrid vs Reranked

In [ ]:
# Build comparison table
if comparison_results:
    df_comparison = pd.DataFrame(comparison_results)
    
    print(f"\nStage Comparison (Top 1 Result):")
    print(df_comparison[['query_idx', 'hybrid_top1_source', 'reranked_top1_source', 'hybrid_top1_score', 'reranked_top1_score']].to_string())
    
    # Count stage-wise changes
    changed = (df_comparison['hybrid_top1_source'] != df_comparison['reranked_top1_source']).sum()
    print(f"\nTop-1 results changed by reranking: {changed}/{len(comparison_results)}")
else:
    print("No comparison data available")

## 12. Save Reranker Results

In [ ]:
# Reorder columns for clarity
column_order = [
    'rank', 'reranker_score', 'hybrid_score', 'dense_score', 'bm25_score',
    'chunk_id', 'doc_id',
    'source', 'category',
    'question', 'answer',
    'chunk_text',
    'test_query_index', 'test_query_text'
]
return_cols = [c for c in column_order if c in df_reranked.columns]
df_reranked = df_reranked[return_cols]

print(f"✓ Finalized reranked results DataFrame")
print(f"  Shape: {df_reranked.shape}")

In [ ]:
# Save as CSV
csv_output = f"{RERANKER_OUTPUT_DIR}/reranked_test_results.csv"
df_reranked.to_csv(csv_output, index=False)
print(f"✓ Saved: {csv_output}")
print(f"  Size: {Path(csv_output).stat().st_size / 1024:.2f} KB")

In [ ]:
# Save as JSONL
jsonl_output = f"{RERANKER_OUTPUT_DIR}/reranked_test_results.jsonl"
with open(jsonl_output, 'w', encoding='utf-8') as f:
    for idx, row in df_reranked.iterrows():
        json_record = row.to_dict()
        json_record = {
            k: (None if pd.isna(v) else v) for k, v in json_record.items()
        }
        f.write(json.dumps(json_record, ensure_ascii=False) + "\n")

print(f"✓ Saved: {jsonl_output}")
print(f"  Size: {Path(jsonl_output).stat().st_size / 1024:.2f} KB")

In [ ]:
# Save metadata
metadata = {
    'corpus_size': len(df_corpus),
    'embedding_model': EMBEDDING_MODEL_NAME,
    'reranker_model': RERANKER_MODEL_NAME,
    'top_k_final': TOP_K_FINAL,
    'hybrid_candidates': HYBRID_CANDIDATES,
    'dense_candidates': DENSE_CANDIDATES,
    'bm25_candidates': BM25_CANDIDATES,
    'alpha': ALPHA,
    'test_queries_count': len(test_queries),
    'total_results': len(df_reranked),
    'stage_comparison_top1_changed': int((df_comparison['hybrid_top1_source'] != df_comparison['reranked_top1_source']).sum()) if comparison_results else 0,
    'files': {
        'reranked_test_results_csv': 'reranked_test_results.csv',
        'reranked_test_results_jsonl': 'reranked_test_results.jsonl',
        'reranker_metadata': 'reranker_metadata.json',
        'reranker_stage_comparison': 'reranker_stage_comparison.csv'
    }
}

metadata_path = f"{RERANKER_OUTPUT_DIR}/reranker_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Saved: {metadata_path}")

In [ ]:
# Save comparison table
if comparison_results:
    comparison_path = f"{RERANKER_OUTPUT_DIR}/reranker_stage_comparison.csv"
    df_comparison.to_csv(comparison_path, index=False)
    print(f"✓ Saved comparison: {comparison_path}")

## 13. Final Summary and Next Steps

In [ ]:
print("\n" + "="*80)
print("RERANKER BASELINE COMPLETE")
print("="*80)

print(f"\nRetrieval Pipeline Summary:")
print(f"  • Corpus size: {len(df_corpus):,} chunks")
print(f"  • Dense embedding model: {EMBEDDING_MODEL_NAME}")
print(f"  • Cross-encoder reranker: {RERANKER_MODEL_NAME}")
print(f"  • Note: Reranker is trained on English MS MARCO (not Turkish-legal-specific)")

print(f"\nReranking Configuration:")
print(f"  • Top K final: {TOP_K_FINAL}")
print(f"  • Hybrid candidate pool: {HYBRID_CANDIDATES}")
print(f"  • Dense contribution (alpha): {ALPHA}")
print(f"  • BM25 contribution: {1 - ALPHA}")

print(f"\nTest Results:")
print(f"  • Test queries: {len(test_queries)}")
print(f"  • Total reranked results: {len(df_reranked)}")
print(f"  • Results per query: {len(df_reranked) // len(test_queries)}")

if comparison_results:
    changed = (df_comparison['hybrid_top1_source'] != df_comparison['reranked_top1_source']).sum()
    print(f"\nReranking Impact:")
    print(f"  • Top-1 results changed: {changed}/{len(comparison_results)}")
    print(f"  • Stage-wise consistency: {len(comparison_results) - changed}/{len(comparison_results)}")

print(f"\nOutput Files (saved to {RERANKER_OUTPUT_DIR}):")
output_files = [
    ("reranked_test_results.csv", "Reranked results for 5 test queries"),
    ("reranked_test_results.jsonl", "Same results in JSONL format"),
    ("reranker_metadata.json", "Metadata about reranker configuration"),
    ("reranker_stage_comparison.csv", "Comparison: hybrid vs reranked top-1 results"),
]

for fname, desc in output_files:
    path = f"{RERANKER_OUTPUT_DIR}/{fname}"
    if Path(path).exists():
        size = Path(path).stat().st_size
        size_str = f"{size / (1024*1024):.2f} MB" if size > 1024*1024 else f"{size / 1024:.2f} KB"
        print(f"  ✓ {fname}")
        print(f"    {desc} ({size_str})")

print(f"\nArchitecture Progress:")
print(f"  ✓ Step 1: Dataset preparation")
print(f"  ✓ Step 2: Retrieval corpus preparation")
print(f"  ✓ Step 3: Dense retrieval baseline")
print(f"  ✓ Step 4: Hybrid retrieval baseline")
print(f"  ✓ Step 5: Reranker baseline (this notebook)")
print(f"  → Step 6: LLM answer generation")
print(f"  → Step 7: End-to-end RAG evaluation benchmark")

print(f"\nNext Steps:")
print(f"  1. Integrate LLM for answer generation (notebook 06)")
print(f"  2. Create benchmark evaluation suite")
print(f"  3. Evaluate retrieval quality (NDCG, MRR, Recall@K)")
print(f"  4. Evaluate answer quality (BLEU, ROUGE, exact match)")
print(f"  5. Build Gradio interface for interactive demo")
print(f"  6. Deploy as API endpoint")

print(f"\nNotes:")
print(f"  • Reranker is NOT Turkish-legal-specific (trained on English MS MARCO)")
print(f"  • Replace with stronger/domain-adapted reranker for production")
print(f"  • Current setup is for baseline comparison only")
print(f"  • Next step: integrate LLM for answer generation")
print(f"  • Then: formal benchmark evaluation of full RAG pipeline")